In [1]:
import os
import sys
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("WARNING: GPU is not available.")
    print("For this research model, Kaggle GPU is strongly recommended.")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8


In [2]:
!pip -q install timm scikit-learn seaborn

# Install/Import Required Libraries #  

In [3]:
import os
import json
import time
import math
import copy
import random
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    balanced_accuracy_score
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import torchvision
from torchvision import transforms

import timm

warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("timm:", timm.__version__)

Device: cuda
Torch: 2.10.0+cu128
Torchvision: 0.25.0+cu128
timm: 1.0.26


In [4]:
SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    os.environ["PYTHONHASHSEED"] = str(seed)
    
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(SEED)

print("Random seed:", SEED)

Random seed: 42


In [5]:
# ============================================================
# CONFIGURATION
# ============================================================

IMAGE_SIZE = 224

BATCH_SIZE = 16          # Reduce to 8 if GPU memory is insufficient
NUM_WORKERS = 2

EPOCHS = 15
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

PATIENCE = 4

# CNN backbone
CNN_BACKBONE = "resnet50"

# Transformer backbone
SWIN_BACKBONE = "swin_tiny_patch4_window7_224"

# Feature dimension after projection
FUSION_DIM = 192

DROPOUT = 0.30

# Mixed precision
USE_AMP = torch.cuda.is_available()

# Number of classes
NUM_CLASSES = None

# Save directory
OUTPUT_DIR = Path("/kaggle/working/plant_disease_stcfi")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_DIR)

Output directory: /kaggle/working/plant_disease_stcfi


In [6]:
CNN_BACKBONE = "resnet101"

In [7]:
INPUT_DIR = Path("/kaggle/input")

print("Datasets available in Kaggle:")
for p in INPUT_DIR.iterdir():
    print(" -", p)

Datasets available in Kaggle:
 - /kaggle/input/datasets


In [8]:
def find_dataset_dirs(keyword):
    keyword = keyword.lower()
    matches = []
    
    for p in INPUT_DIR.rglob("*"):
        if p.is_dir() and keyword in p.name.lower():
            matches.append(p)
    
    return matches

plantvillage_candidates = find_dataset_dirs("plant")
print("\nPossible plant datasets:")

for p in plantvillage_candidates[:30]:
    print(p)


Possible plant datasets:
/kaggle/input/datasets/andresmgs/plantdec
/kaggle/input/datasets/abdallahalidev/plantvillage-dataset
/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/plantvillage dataset


# Locate PlantVillage and PlantDoc # 

In [9]:
for p in INPUT_DIR.iterdir():
    print("\nDATASET:", p)
    if p.is_dir():
        try:
            for child in list(p.iterdir())[:20]:
                print("   ", child.name)
        except Exception:
            pass


DATASET: /kaggle/input/datasets
    andresmgs
    abdallahalidev


In [10]:
# ============================================================
# SET DATASET PATHS AFTER INSPECTING CELL 6
# ============================================================

PLANTVILLAGE_ROOT = Path("/kaggle/input/PlantVillage Dataset")

# Change this if your PlantDoc folder has a different name.
PLANTDOC_ROOT = Path("/kaggle/input/PlantDOC")

print("PlantVillage:", PLANTVILLAGE_ROOT)
print("Exists:", PLANTVILLAGE_ROOT.exists())

print("PlantDoc:", PLANTDOC_ROOT)
print("Exists:", PLANTDOC_ROOT.exists())

PlantVillage: /kaggle/input/PlantVillage Dataset
Exists: False
PlantDoc: /kaggle/input/PlantDOC
Exists: False
